# Setup

To run this notebook on Colab, a few setup steps are required. Follow along step by step:

1. **Clone the `dlfb` library**  
   First, clone the repository that contains the `dlfb` library.

In [ ]:
%cd /content
!rm -rf ./dlfb-pytorch-clone/
!git clone "https://github.com/deep-learning-for-biology/deep-learning-for-biology-pytorch.git" dlfb-pytorch-clone --branch main
%cd dlfb-pytorch-clone

2. **Install dependencies**  
   Once the library is cloned, install the required dependencies.

In [ ]:
%%bash
pip install -e '.[graphs]'

3. **Providion the datasets**  
   You’ll then need to access and download the necessary datasets for this chapter.

In [ ]:
from google.colab import auth

auth.authenticate_user()
# NOTE: exclude models with '--no-models' flag
!dlfb-provision --chapter graphs

4. **Load the `dlfb` package**  
   Finally, load the `dlfb` package.  
   - ⚠️ Note: Loading can sometimes be finicky. If you encounter issues, simply **restart the runtime**. All previously downloaded data and installed packages will persist, so you can re-run the load step without repeating everything.

In [ ]:
try:
  import dlfb_pytorch
except ImportError as exc:
  import site
  site.main()
  import dlfb_pytorch

import inspect
from dlfb_pytorch.utils.display import display

# 4. Understanding Drug–Drug Interactions Using Graphs


## 4.1. Biology Primer
### 4.1.1. Beneficial Drug–Drug Interactions
### 4.1.2. Harmful Drug–Drug Interactions
### 4.1.3. DrugBank


## 4.2. Machine Learning Primer
### 4.2.1. Representing Graph Structures
### 4.2.2. Graph Neural Networks
### 4.2.3. Graph Embeddings and Message Passing
### 4.2.4. Cold-Start Problem
### 4.2.5. GraphSAGE


## 4.3. Selecting a Dataset
### 4.3.1. Describing the Dataset
### 4.3.2. Exploring the Dataset


In [ ]:
from ogb.linkproppred import LinkPropPredDataset

from dlfb_pytorch.utils.context import assets

# Quite a large graph, may take a few minutes to load.
dataset = LinkPropPredDataset(name="ogbl-ddi", root=assets("graphs/datasets"))

In [ ]:
dataset.graph

In [ ]:
print(
  f'The graph contains {dataset.graph["num_nodes"]} nodes and '
  f'{dataset.graph["edge_index"].shape[1]} edges.'
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

degrees = np.bincount(dataset.graph["edge_index"].flatten())

sns.histplot(degrees, kde=True)
plt.xlabel("Degree")
plt.ylabel("Frequency")
plt.title("Degree Distribution");

In [ ]:
num_nodes = dataset.graph["num_nodes"]
num_observed_edges = dataset.graph["edge_index"].shape[1]

# Since each edge in an undirected graph can be represented in two ways, we
# multiply by 2 to account for the bidirectionality.
num_observed_edges = 2 * num_observed_edges

# For any graph with n nodes, the maximum number of edges (assuming no
# self-loops) is n * (n-1).
num_possible_edges = num_nodes * (num_nodes - 1)

density = num_observed_edges / num_possible_edges

print(
  f"There are {num_observed_edges} observed edges and {num_possible_edges} "
  f"possible edges,\ngiving a graph density of {round(density, 2)}."
)

In [ ]:
data_split = dataset.get_edge_split()
data_split

In [ ]:
print(
  f'Number of edges in train set: {data_split["train"]["edge"].shape[0]}\n'
  f'Number of edges in valid set: {data_split["valid"]["edge"].shape[0]}\n'
  f'Number of edges in test set: {data_split["test"]["edge"].shape[0]}'
)

In [ ]:
train_nodes = np.unique(data_split["train"]["edge"])
valid_nodes = np.unique(data_split["valid"]["edge"])
test_nodes = np.unique(data_split["test"]["edge"])

# Check if all nodes in valid and test sets are present in train set.
valid_in_train = np.isin(valid_nodes, train_nodes).all()
test_in_train = np.isin(test_nodes, train_nodes).all()

print(f"All validation nodes are in training nodes: {valid_in_train}")
print(f"All test nodes are in training nodes: {test_in_train}")

### 4.3.3. Examining Drug Names


In [ ]:
import pandas as pd

ddi_descriptions = pd.read_csv(
  assets("graphs/datasets/ogbl_ddi/mapping/ddi_description.csv.gz")
)
print(ddi_descriptions)

In [ ]:
node_to_dbid_lookup = pd.read_csv(
  assets("graphs/datasets/ogbl_ddi/mapping/nodeidx2drugid.csv.gz")
)
print(node_to_dbid_lookup)

In [ ]:
pd.set_option("display.max_rows", 100)
pd.set_option("display.min_rows", 100)

In [ ]:
ddi_descriptions["first drug name"].value_counts().head(10)

In [ ]:
first_drug = ddi_descriptions[["first drug id", "first drug name"]].rename(
  columns={"first drug id": "dbid", "first drug name": "drug_name"}
)
second_drug = ddi_descriptions.loc[
  :, ["second drug id", "second drug name"]
].rename(columns={"second drug id": "dbid", "second drug name": "drug_name"})
dbid_to_name_lookup = (
  pd.concat([first_drug, second_drug]).drop_duplicates().reset_index(drop=True)
)

drugs_lookup = pd.merge(
  node_to_dbid_lookup.rename(
    columns={"drug id": "dbid", "node idx": "node_id"}
  ),
  dbid_to_name_lookup,
  on="dbid",
  how="inner",
)

drugs_lookup.iloc[935]

### 4.3.4. Visualizing Graphs


In [ ]:
import numpy as np

np.random.seed(42)


def get_subgraph(edges: np.ndarray, node_limit: int) -> np.ndarray:
  """Gets a subgraph by sampling nodes and their edges."""
  nodes = np.unique(edges)
  sampled_nodes = np.random.choice(nodes, size=node_limit, replace=False)
  filtered_edges = edges[
    np.isin(edges[:, 0], sampled_nodes) & np.isin(edges[:, 1], sampled_nodes)
  ]
  print(f"Subgraph has {filtered_edges.shape[0]} edges")
  return filtered_edges


# Sample 50 nodes from the training set.
subgraph = get_subgraph(node_limit=50, edges=data_split["train"]["edge"])

In [ ]:
import networkx as nx
from adjustText import adjust_text


def plot_ddi_graph(graph: np.ndarray, drugs_lookup: pd.DataFrame) -> plt.Figure:
  """Plots a drug–drug interaction graph with labeled nodes."""
  fig = plt.figure(figsize=(15, 15))
  G = nx.Graph()
  G.add_edges_from(graph)
  pos = nx.spring_layout(G)
  nx.draw(
    G=G,
    pos=pos,
    with_labels=False,
    node_color="lightgray",
    edge_color="gray",
    node_size=10,
    alpha=0.3,
  )
  names = (
    drugs_lookup[drugs_lookup["node_id"].isin(G.nodes)]
    .set_index("node_id")["drug_name"]
    .to_dict()
  )
  labels = nx.draw_networkx_labels(G=G, pos=pos, labels=names, font_size=20)
  adjust_text(list(labels.values()))
  return fig


plot_ddi_graph(subgraph, drugs_lookup);

## 4.4. Building a Dataset
### 4.4.1. Creating a Dataset Builder


In [ ]:
from dlfb_pytorch.graphs.dataset.builder import DatasetBuilder

display([DatasetBuilder.__init__, DatasetBuilder.build])

### 4.4.2. Download the Raw Dataset


In [ ]:
display([DatasetBuilder.download])

### 4.4.3. Prepare the Annotation


In [ ]:
display([DatasetBuilder.prepare_annotation])

### 4.4.4. Prepare the Graph


In [ ]:
display([DatasetBuilder.prepare_graph])

In [ ]:
display([DatasetBuilder.make_undirected])

### 4.4.5. Prepare the Pairs


In [ ]:
display([DatasetBuilder.prepare_pairs])

In [ ]:
display([DatasetBuilder.infer_negative_pairs])

### 4.4.6. Subsetting the Graph


In [ ]:
display([DatasetBuilder.subset])

### 4.4.7. The Dataset Class


In [ ]:
import re

from dlfb_pytorch.graphs.dataset import Dataset

display([re.split(r"\n\s+def\s", inspect.getsource(Dataset))[0]])

## 4.5. Building a Prototype
### 4.5.1. Node Encoder


In [ ]:
from dlfb_pytorch.graphs.model import NodeEncoder

display([NodeEncoder])

### 4.5.2. Graph Convolution


In [ ]:
from dlfb_pytorch.graphs.model import SAGEConv

display([SAGEConv])

#### 4.5.2.1. Adding Self Edges
#### 4.5.2.2. Aggregating the Neighborhood
#### 4.5.2.3. Normalizing by Degree
#### 4.5.2.4. Combining Embeddings with Neighborhoods
### 4.5.3. Link Prediction


In [ ]:
from dlfb_pytorch.graphs.model import LinkPredictor

display([LinkPredictor])

### 4.5.4. Drug–Drug Interaction Model


In [ ]:
from dlfb_pytorch.graphs.model import DdiModel

display([DdiModel])

## 4.6. Training the Model
### 4.6.1. Create a Manageable Dataset


In [ ]:
import torch

node_limit = 500
torch.manual_seed(42)

dataset_splits = DatasetBuilder(path=assets("graphs/datasets")).build(
  node_limit,
)

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 100

In [ ]:
from dlfb_pytorch.graphs.inspect import plot_graph

plot_graph(dataset_splits["train"]);

### 4.6.2. The Training Loop


In [ ]:
from dlfb_pytorch.graphs.train import train

display([train])

### 4.6.3. The Pairs Class


In [ ]:
from dlfb_pytorch.graphs.dataset.pairs import Pairs

display([Pairs])

#### 4.6.3.1. Batching by Pairs


In [ ]:
from dlfb_pytorch.graphs.train import optimal_batch_size

display([optimal_batch_size])

#### 4.6.3.2. Sampling Negative Pairs


In [ ]:
display([Pairs._global_negative_sampling])

### 4.6.4. Create the Train Step Function


In [ ]:
from dlfb_pytorch.graphs.train import train_step

display([train_step])

In [ ]:
from dlfb_pytorch.graphs.train import binary_log_loss

display([binary_log_loss])

### 4.6.5. Create the Evaluation Metric


In [ ]:
from dlfb_pytorch.graphs.train import eval_step

display([eval_step])

In [ ]:
from dlfb_pytorch.graphs.train import evaluate_hits_at_20

display([evaluate_hits_at_20])

### 4.6.6. Train the Simplest Model


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = DdiModel(
  n_nodes=dataset_splits["train"].n_nodes,
  embedding_dim=128,
  last_layer_self=False,
  degree_norm=False,
  dropout_rate=0.3,
).to(device)

optimizer = model.create_optimizer(lr=0.001)

model, optimizer, metrics = train(
  model=model,
  optimizer=optimizer,
  dataset_splits=dataset_splits,
  num_epochs=500,
  eval_every=1,
  loss_fn=binary_log_loss,
  norm_loss=False,
  store_path=assets("graphs/models/initial_model"),
)

In [ ]:
from dlfb_pytorch.graphs.inspect import plot_learning

plot_learning(metrics);

## 4.7. Improving the Model
### 4.7.1. Change to AUC Loss


In [ ]:
from dlfb_pytorch.graphs.train import auc_loss

display([auc_loss])

In [ ]:
torch.manual_seed(42)

model = DdiModel(
  n_nodes=dataset_splits["train"].n_nodes,
  embedding_dim=128,
  last_layer_self=False,
  degree_norm=False,
  dropout_rate=0.3,
).to(device)

optimizer = model.create_optimizer(lr=0.001)

_, _, metrics = train(
  model=model,
  optimizer=optimizer,
  dataset_splits=dataset_splits,
  num_epochs=500,
  eval_every=1,
  loss_fn=auc_loss,
  norm_loss=True,
  store_path=assets("graphs/models/initial_model_auc"),
)

In [ ]:
plot_learning(metrics);

### 4.7.2. Set Model Sweeping and Training Parameters
#### 4.7.2.1. Varying Embedding Dimensions


In [ ]:
embedding_dims = [64, 128, 256, 512]

model_params = {
  "n_nodes": dataset_splits["train"].n_nodes,
  "last_layer_self": False,
  "degree_norm": False,
  "dropout_rate": 0.3,
}
training_params = {
  "rng": rng_train,
  "dataset_splits": dataset_splits,
  "num_epochs": 500,
  "eval_every": 25,
  "loss_fn": auc_loss,
  "norm_loss": True,
}

In [ ]:
from dlfb_pytorch.utils.metric_plots import to_df

all_metrics = []

for embedding_dim in embedding_dims:
  torch.manual_seed(42)
  model = DdiModel(**{"embedding_dim": embedding_dim, **model_params}).to(device)
  optimizer = model.create_optimizer(lr=0.001)
  _, _, metrics = train(
    model=model,
    optimizer=optimizer,
    **training_params,
    store_path=assets(f"graphs/models/sweep_embedding_dim:{embedding_dim}"),
  )

  df = to_df(metrics).assign(**{"embedding_dim": embedding_dim})
  all_metrics.append(df)

all_metrics_df = pd.concat(all_metrics, axis=0)

In [ ]:
import seaborn as sns
from matplotlib import pyplot as plt

from dlfb_pytorch.utils.metric_plots import DEFAULT_SPLIT_COLORS

data = all_metrics_df
data = data[(data["metric"] == "hits@20")]
data = data.groupby(["metric", "split", "embedding_dim"], as_index=False)[
  "mean"
].max()
data = data.sort_values(by=["split", "mean"])

plt.figure(figsize=(7, 3.5))
sns.barplot(
  data=data,
  x="embedding_dim",
  y="mean",
  hue="split",
  palette=DEFAULT_SPLIT_COLORS,
)
plt.ylim(0.5, 1)
plt.xlabel("Embedding Dimension")
plt.ylabel("Hits@20")
plt.legend(title="Split")
plt.tight_layout()

#### 4.7.2.2. Varying multiple hyperparameters


In [ ]:
import itertools

model_params = {
  "n_nodes": dataset_splits["train"].n_nodes,
  "embedding_dim": 512,
}

model_params_sweep = {
  "dropout_rate": [0, 0.3, 0.5],
  "last_layer_self": [True, False],
  "degree_norm": [True, False],
  "n_mlp_layers": [1, 2, 3],
}
keys, values = zip(*model_params_sweep.items())
model_param_combn = [
  dict(zip(keys, combo)) for combo in itertools.product(*values)
]
print(pd.DataFrame(model_param_combn))

In [ ]:
def name_from_params(params: dict) -> str:
  """Generates a string from a parameters dictionary"""
  return "_".join([f"{k}:{v}" for k, v in params.items()])


all_metrics = []

for combn in model_param_combn:
  torch.manual_seed(42)
  model = DdiModel(**{**combn, **model_params}).to(device)
  optimizer = model.create_optimizer(lr=0.001)
  _, _, metrics = train(
    model=model,
    optimizer=optimizer,
    **training_params,
    store_path=assets(f"graphs/models/sweep_all_{name_from_params(combn)}"),
  )

  df = to_df(metrics).assign(**combn)
  all_metrics.append(df)

all_metrics_df = pd.concat(all_metrics, axis=0)

In [ ]:
def conv_layer_annot(row):
  if row["last_layer_self"] and row["degree_norm"]:
    return "with self-edges and norm"
  elif row["last_layer_self"]:
    return "with self-edges, no norm"
  elif row["degree_norm"]:
    return "with norm, no self-edges"
  else:
    return "no self-edges and no norm"


data = all_metrics_df
data = data[(data["metric"] == "hits@20")]
data = data.groupby(
  ["metric", "split", *list(model_params_sweep.keys())], as_index=False
)["mean"].max()
data = data.sort_values(by=["split", "mean"])
data["conv_layer"] = data.apply(conv_layer_annot, axis=1)

In [ ]:
fig = sns.relplot(
  data=data,
  x="conv_layer",
  y="mean",
  row="dropout_rate",
  col="n_mlp_layers",
  hue="split",
  palette=DEFAULT_SPLIT_COLORS,
  facet_kws=dict(margin_titles=True, despine=False),
  height=2,
)
fig.figure.subplots_adjust(wspace=0.1, hspace=0)
for ax in fig.axes.flat:
  for label in ax.get_xticklabels():
    label.set_rotation(45)
    label.set_ha("right")
fig.set_axis_labels("", "maximum Hits@20");

In [ ]:
from dlfb_pytorch.utils.metric_plots import from_df

metrics = all_metrics_df
metrics = from_df(
  metrics[
    (metrics["dropout_rate"] == 0.0)
    & (metrics["last_layer_self"] == False)
    & (metrics["degree_norm"] == False)
    & (metrics["n_mlp_layers"] == 1)
  ]
)

plot_learning(metrics);

In [ ]:
metrics = all_metrics_df
metrics = from_df(
  metrics[
    (metrics["dropout_rate"] == 0.5)
    & metrics["last_layer_self"]
    & (metrics["degree_norm"] == False)
    & (metrics["n_mlp_layers"] == 2)
  ]
)

plot_learning(metrics);

### 4.7.3. Train on a Larger Dataset


In [ ]:
node_limit = 2134
torch.manual_seed(42)
dataset_splits = DatasetBuilder(path=assets("graphs/datasets")).build(
  node_limit,
)

model = DdiModel(
  n_nodes=dataset_splits["train"].n_nodes,
  embedding_dim=512,
  dropout_rate=0.3,
  last_layer_self=True,
  degree_norm=True,
  n_mlp_layers=2,
).to(device)

optimizer = model.create_optimizer(lr=0.001)

_, _, metrics = train(
  model=model,
  optimizer=optimizer,
  dataset_splits=dataset_splits,
  num_epochs=1000,
  eval_every=25,
  loss_fn=auc_loss,
  norm_loss=True,
  store_path=assets("graphs/models/larger_model"),
)

In [ ]:
plot_learning(metrics);

### 4.7.4. Extensions


## 4.8. Summary
